## 结构化输出

结构化输出是一种约束模型生成特定格式的方法，使用 JSON Schema 或 Zod 来定义输出结构。与传统的自由文本输出相比，结构化输出有以下优势：

### 优势
- **可预测性**：模型输出遵循预定义的结构，不会产生格式不一致的结果
- **易于解析**：JSON 格式的输出可以直接被程序解析，无需手动提取
- **类型安全**：使用 Zod 等库可以验证输出数据类型和约束
- **更好的集成**：结构化输出可以直接用于下游应用，减少数据处理的复杂性

### 使用场景
- 数据提取：从文本中提取结构化的信息
- 分类任务：将输入分类到预定义的类别中
- 生成列表：创建具有特定字段的对象列表
- 表单填充：基于用户输入生成完整的表单数据

### 实现方法
使用 OpenAI `zodTextFormat` 辅助函数配合 Zod Schema 定义输出格式。

In [1]:
import { OpenAI } from 'openai'
import { zodTextFormat } from 'openai/helpers/zod'
import { z } from 'zod'

const client = new OpenAI()


基础示例

In [ ]:
// 定义输出结构：书籍信息
const BookInfoSchema = z.object({
  title: z.string().describe('书籍名称'),
  author: z.string().describe('作者名称'),
  genre: z.string().describe('书籍类型'),
  summary: z.string().describe('内容简介'),
  keyPoints: z.array(z.string()).describe('关键要点'),
  rating: z.number().min(1).max(10).describe('评分（1-10）'),
})

// 使用结构化输出提取书籍信息
const response = await client.responses.parse({
  model: 'gpt-5.4-nano',
  input: [
    {
      role: 'user',
      content: `请从以下文本中提取书籍信息，并按照指定的JSON格式返回：

《球状闪电》是刘慈欣的科幻小说，是《三体》的前传。小说讲述了一个少年因为离奇的雷雨事件失去双亲，从而踏上研究球状闪电的旅程。书中展现了对物理学、人生哲学的深入思考，拥有狂野的想象力，将读者从另一个维度观察世界。这是刘慈欣三大长篇之一，与《三体》和《超新星纪元》齐名。

用户对此书的评价：设想大胆、构思精妙、剧情曲折、人物深刻、令人深思。`,
    },
  ],
  text: {
    format: zodTextFormat(BookInfoSchema, 'event'),
  },
})

console.log('API 响应内容：', response.output_parsed)


API 响应内容： {
  title: "球状闪电",
  author: "刘慈欣",
  genre: "科幻小说",
  summary: "《球状闪电》是刘慈欣的科幻小说，被认为是《三体》的前传。故事讲述一名少年因离奇雷雨事件失去双亲，因而踏上研究球状闪电的旅程。作品体现了对物理学与人生哲学的深度思考，想象力大胆奇特，引导读者从另一个维度理解世界。",
  keyPoints: [
    "刘慈欣科幻长篇",
    "被称为《三体》的前传",
    "少年因离奇雷雨失去双亲，开启研究球状闪电之旅",
    "对物理学与人生哲学有深入思考",
    "想象力大胆，叙事引人深思",
    "评价要点：设想大胆、构思精妙、剧情曲折、人物深刻、令人深思",
    "与《三体》《超新星纪元》并列为刘慈欣三大长篇之一"
  ],
  rating: 9
}


In [ ]:
const Step = z.object({
  explanation: z.string(),
  output: z.string(),
})

const MathReasoning = z.object({
  steps: z.array(Step),
  final_answer: z.string(),
})

const response1 = await client.responses.parse({
  model: 'gpt-5.4-nano',
  input: [
    {
      role: 'system',
      content: '你是一位数学辅导助手。请逐步引导用户完成解题过程。',
    },
    { role: 'user', content: '如何解方程 8x + 7 = -23' },
  ],
  text: {
    format: zodTextFormat(MathReasoning, 'math_reasoning'),
  },
})

const math_reasoning = response1.output_parsed
console.log(math_reasoning)


{
  steps: [
    {
      explanation: "先把含 x 的项 8x 单独拿出来：两边同时减去 7。",
      output: "8x + 7 = -23\n8x + 7 - 7 = -23 - 7\n8x = -30"
    },
    {
      explanation: "接着把 x 的系数 8 去掉：两边同时除以 8。",
      output: "8x = -30\n8x/8 = -30/8\nx = -30/8"
    },
    { explanation: "最后把分数约分成最简形式。", output: "x = -30/8 = -15/4" }
  ],
  final_answer: "方程 8x + 7 = -23 的解是 x = -15/4。"
}


In [3]:
async function getInvoiceInformation(input: string | OpenAI.Responses.ResponseInput) {
  const InvoiceSchema = z.object({
    no: z.string().describe('发票编号'),
    money: z.number().describe('金额'),
    project: z.string().describe('项目'),
    endDate: z.string().describe('到期时间'),
  })
  const res = await client.responses.parse({
    model: 'gpt-5.4-nano',
    input,
    text: {
      format: zodTextFormat(InvoiceSchema, 'invoice'),
    },
  })
  return res
}
const res2 = await getInvoiceInformation('发票 123，3 个小部件 45.99 美元，3 月 30 日到期')
console.log(res2.output_parsed)


{ no: "123", money: 45.99, project: "3 个小部件", endDate: "3月30日" }
